# Conditioned Earnings Event Study (Multi-Source PEAD)

    Earnings surprises and revisions combined with macro release context, liquidity regime and short-interest to produce conditioned post-earnings drift analysis across many names.

    **Category:** Multi-source event study / alpha research

    **Primary API calls used:**
    - Earnings calendar + surprises (equity.calendar, fmp)
    - Fundamentals + estimates / revisions (equity.fundamentals, equity.estimates)
    - Macro calendar / releases around event window
    - Pricing + volume + short interest (microstructure)

    Public data only. No internal book required.

## Run Output

![73_conditioned_earnings_pead_multi_source](../plots/73_conditioned_earnings_pead_multi_source_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
import os
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantjourney.sdk import QuantJourneyAPI

qj = QuantJourneyAPI.from_env()

START = os.getenv("QJ_EXAMPLE_START", "2019-01-01")
END = os.getenv("QJ_EXAMPLE_END", "2026-06-06")

plt.style.use("default")
plt.rcParams.update({"figure.figsize": (12, 4.5), "axes.grid": True})


def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and "data" in payload: payload = payload["data"]
    if isinstance(payload, dict) and "value" in payload: return payload["value"]
    return payload


def safe_call(label: str, fn, **kwargs):
    try: return fn(**kwargs)
    except Exception: 
        print(label, "unavailable")
        return None


def as_rows(p): 
    v = unwrap(p)
    if isinstance(v, list): return v
    if isinstance(v, dict):
        for k in ("rows", "data", "items", "earnings"):
            if isinstance(v.get(k), list): return v[k]
        return [v]
    return []


def get_earnings_calendar(start=START, end=END):
    cal = safe_call("earnings calendar", qj.eod.get_earnings_calendar, from_date=start, to_date=end) or \
          safe_call("fmp earnings", qj.fmp.get_earnings_calendar, from_date=start, to_date=end)
    return pd.DataFrame(as_rows(cal))


def get_surprises(symbols):
    out = []
    for s in symbols:
        sup = as_rows(safe_call("surprise " + s, qj.fmp.get_earnings_surprises, symbol=s))
        for r in sup:
            out.append({"symbol": s, **r})
    return pd.DataFrame(out)


def price_around(symbol, event_date, window=21):
    p = safe_call("px " + symbol, qj.eod.get_historical_prices, symbol=symbol, start_date=str(event_date - pd.Timedelta(days=40)), end_date=str(event_date + pd.Timedelta(days=40)))
    df = pd.DataFrame(as_rows(p))
    if df.empty: return None
    df["date"] = pd.to_datetime(df.get("date"))
    df = df.set_index("date").sort_index()
    price = df.get("adjusted_close").fillna(df.get("close"))
    if event_date not in price.index: return None
    idx = price.index.get_loc(event_date)
    if idx - window < 0 or idx + window >= len(price): return None
    pre = price.iloc[idx - 5 : idx].mean()
    post = price.iloc[idx : idx + 21]
    return ((post / pre) - 1).reset_index()


In [ ]:
universe = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META"]

cal = get_earnings_calendar()
print("Earnings calendar rows:", len(cal))

surp = get_surprises(universe)
print("Surprise rows:", len(surp))

# Simple reaction collection (toy)
reactions = []
for _, row in (surp.head(30) if not surp.empty else pd.DataFrame()).iterrows():
    sym = row.get("symbol")
    ed = pd.to_datetime(row.get("date") or row.get("reportDate"))
    if pd.isna(ed): continue
    r = price_around(sym, ed)
    if r is not None:
        r["symbol"] = sym
        reactions.append(r)

if reactions:
    react_df = pd.concat(reactions)
    print("\nAverage cumulative post-event (toy sample):")
    print(react_df.groupby("symbol").tail(1)[[0]].mean())
    react_df.groupby("symbol").mean().plot(title="Average post-earnings path (toy)")
    plt.show()

print("\nExtend this candidate with macro release overlay and liquidity buckets for true multi-source conditioning.")

## Notes

Multi-source event study candidate: earnings calendar + surprises/revisions + pricing + (extend with macro releases and microstructure).
All data is public. Scale the universe and add proper surprise standardization + macro date alignment for production use.